# 📈 Exploración de Fitness y Métricas Deportivas (v2.0)

Este cuaderno carga el historial de TSS (Training Stress Score) para visualizar la evolución del **PMC (Performance Management Chart)**: Fitness, Fatiga y Forma de los últimos 90 días.

In [1]:
import os
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output

# Añadir el raíz del proyecto al path para importar src.*
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_PATH not in sys.path:
    sys.path.insert(0, ROOT_PATH)

from src.core.pmc import PMCProcessor
from src.core.athlete import AthleteProfile
from src.core.analytics import ActivityMetricsCalculator
from src.api.strava import StravaClient

print("✅ Entorno inicializado correctamente.")

✅ Entorno inicializado correctamente.


## 1. Carga de Datos y Procesamiento Histórico
Para garantizar que el gráfico de carga semanal cubra la ventana completa (12 semanas), forzamos que el índice de procesamiento comience hace 90 días.

In [ ]:
TSS_CACHE_FILE = Path("../data/tss_history.parquet")

def load_processed_data():
    if not TSS_CACHE_FILE.exists():
        print("❌ No se encontró el histórico en data/tss_history.parquet. Por favor, corre sync.py primero.")
        return pd.DataFrame(), pd.DataFrame()
    
    df_raw = pd.read_parquet(TSS_CACHE_FILE)
    df_raw['date'] = pd.to_datetime(df_raw['date'], utc=True).dt.tz_localize(None)
    
    # Definir ventana de 90 días para el gráfico histórico
    hoy = pd.Timestamp.now().normalize()
    inicio_90d = hoy - pd.Timedelta(days=90)
    
    # Padding: Si la primera actividad es reciente, añadimos una fila de relleno al inicio
    df_input = df_raw[['date', 'tss']].copy()
    if df_input['date'].min() > inicio_90d:
        padding = pd.DataFrame({'date': [inicio_90d], 'tss': [0.0]})
        df_input = pd.concat([df_input, padding])
    
    # Calcular PMC completo
    pmc_proc = PMCProcessor()
    df_pmc = pmc_proc.calculate_pmc(df_input)
    
    return df_raw, df_pmc

df_raw, df_pmc = load_processed_data()
if not df_raw.empty:
    print(f"📊 Cargadas {len(df_raw)} actividades y {len(df_pmc)} días de PMC.")

📊 Cargadas 44 actividades y 96 días de PMC.


In [3]:
df_raw[df_raw["tss"]>0].count()

id             44
date           44
name           44
description    44
tss            44
tss_source     44
hr_tss         44
pwr_tss        44
dtype: int64

In [4]:
df_pmc[df_pmc["tss"]>0].count()

date    30
tss     30
ctl     30
atl     30
tsb     30
dtype: int64

## 2. Visualización PMC (Evolución Fitness vs Fatiga)
Interpreta tu **CTL (Azul)** como tu fondo de largo plazo, y el **ATL (Naranja)** como tu fatiga reciente.

In [5]:
def plot_pmc_chart(df):
    # Filtrar solo los últimos 90 días para el gráfico
    hoy = pd.Timestamp.now().normalize()
    df_plot = df[df['date'] >= (hoy - pd.Timedelta(days=90))]
    
    fig = go.Figure()
    
    # CTL (Fitness)
    fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['ctl'], name='Fitness (CTL)', 
                             line=dict(color='royalblue', width=3), fill='tozeroy', 
                             fillcolor='rgba(65, 105, 225, 0.1)'))
    
    # ATL (Fatiga)
    fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['atl'], name='Fatiga (ATL)', 
                             line=dict(color='orange', width=2, dash='dot')))
    
    # TSB (Forma)
    fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['tsb'], name='Forma (TSB)', 
                             line=dict(color='forestgreen', width=1), fill='tozeroy', 
                             fillcolor='rgba(34, 139, 34, 0.1)'))
    
    fig.update_layout(title='Performance Management Chart (Últimos 90 días)', 
                      xaxis_title='Fecha', yaxis_title='Puntos TSS',
                      template='plotly_dark', legend_orientation='h')
    fig.show()

if not df_pmc.empty:
    plot_pmc_chart(df_pmc)

## 3. Carga Semanal de Entrenamiento
Visualiza tus bloques de carga acumulada por semana (Últimas 12 semanas).

In [6]:
def plot_weekly_bars(df):
    # Agrupar por semana usando el resample de pandas
    df_weekly = df.set_index('date').resample('W').sum().tail(12)
    
    fig = px.bar(df_weekly, x=df_weekly.index, y='tss', 
                 title='Carga Acumulada Semanal (TSS)',
                 labels={'tss': 'Suma TSS', 'date': 'Semana'},
                 template='plotly_dark', color='tss', color_continuous_scale='Magma')
    
    fig.update_layout(coloraxis_showscale=False)
    fig.show()

if not df_pmc.empty:
    plot_weekly_bars(df_pmc)

## 4. Analizador Interactivo de Sesiones
Selecciona una actividad de tus últimos 90 días para ver su telemetría y métricas detalladas.

In [7]:
output_area = widgets.Output()

def analyze_specific_activity(change):
    with output_area:
        clear_output()
        activity_id = change['new']
        if activity_id == 0: return
        
        # metadata básica
        meta = df_raw[df_raw['id'] == activity_id].iloc[0]
        print(f"🔍 Analizando: {meta['name']}")
        print(f"📅 Fecha: {meta['date'].strftime('%Y-%m-%d %H:%M')}")
        print(f"🔥 TSS: {meta['tss']} | Fuente: {meta['tss_source']}")
        print(f"📝 Sensaciones: {meta['description'] or 'Sin comentarios'}")
        print("-" * 50)
        
        # Intentar extraer streams para análisis profundo
        try:
            client = StravaClient()
            athlete = AthleteProfile()
            calc = ActivityMetricsCalculator(athlete)
            
            print("📥 Descargando telemetría de Strava...")
            df_stream = client.get_activity_streams(activity_id, start_date=meta['date'])
            
            if df_stream is not None and not df_stream.empty:
                metrics = calc.process_full_activity_summary(df_stream)
                print("🚀 Métricas Avanzadas Computed:")
                print(f"   - Normal Power: {metrics.get('normalized_power', 'N/A')} W")
                print(f"   - Intensity Factor: {metrics.get('intensity_factor', 'N/A')}")
                print(f"   - Variability Index: {metrics.get('variability_index', 'N/A')}")
                
                # Gráfico de la sesión
                fig = go.Figure()
                if 'watts' in df_stream.columns:
                    fig.add_trace(go.Scatter(x=df_stream.index, y=df_stream['watts'], name='Power (W)', line=dict(color='orange')))
                if 'heartrate' in df_stream.columns:
                    fig.add_trace(go.Scatter(x=df_stream.index, y=df_stream['heartrate'], name='Heart Rate (bpm)', line=dict(color='red')))
                
                fig.update_layout(title=f"Telemetría: {meta['name']}", template='plotly_dark')
                fig.show()
            else:
                print("⚠️ No se encontró telemetría detallada para esta sesión.")
        except Exception as e:
            print(f"❌ Error al analizar sesión: {e}")

# Construir Dropdown con últimas 20 actividades
if not df_raw.empty:
    sorted_acts = df_raw.sort_values('date', ascending=False).head(20)
    dropdown_options = [('Selecciona una actividad...', 0)] + \
                       [(f"{r['date'].strftime('%Y-%m-%d')} - {r['name']}", r['id']) for i, r in sorted_acts.iterrows()]
    
    activity_dropdown = widgets.Dropdown(options=dropdown_options, description='Sesión:', style={'description_width': 'initial'})
    activity_dropdown.observe(analyze_specific_activity, names='value')
    
    display(activity_dropdown)
    display(output_area)

Dropdown(description='Sesión:', options=(('Selecciona una actividad...', 0), ('2026-03-22 - Persiguiendo al am…

Output()